# 02 — Feature Engineering & Treatment Simulation
Build user features and simulate the OTT notification experiment.

In [ ]:
import sys
sys.path.insert(0, "..")
from src.data_loader import load_merged
from src.feature_engineering import build_user_features
from src.simulate_rct import simulate_treatment_outcome
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.1)

## Build per-user feature matrix

In [ ]:
merged = load_merged()
X = build_user_features(merged, split_days=30)
print(X.shape)
X.describe()

## Simulate treatment & outcome — Observational (confounded)

In [ ]:
df_obs = simulate_treatment_outcome(X, mode="observational", seed=42)
print(df_obs[["treatment", "outcome", "tau_true", "propensity"]].describe())

## Simulate treatment & outcome — RCT (sanity check)

In [ ]:
df_rct = simulate_treatment_outcome(X, mode="rct", seed=42)
print(df_rct[["treatment", "outcome", "tau_true"]].describe())

## CATE distribution (ground truth)

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df_obs["tau_true"], bins=50, color="steelblue", edgecolor="white")
ate_val = df_obs["tau_true"].mean()
plt.axvline(ate_val, color="red", linestyle="--", label=f"ATE = {ate_val:.4f}")
plt.legend()
plt.title("Ground-truth CATE distribution")
plt.xlabel("tau(X)")
plt.tight_layout()
plt.savefig("../results/figures/sim_cate_dist.png", bbox_inches="tight")
plt.show()

## Propensity score distribution (observational)

In [ ]:
plt.figure(figsize=(10, 4))
for t_val, grp in df_obs.groupby("treatment"):
    label = "Treated" if t_val == 1 else "Control"
    grp["propensity"].plot(kind="hist", bins=40, alpha=0.7, label=label)
plt.legend()
plt.title("Propensity score by treatment group")
plt.xlabel("P(T=1|X)")
plt.tight_layout()
plt.savefig("../results/figures/sim_propensity.png", bbox_inches="tight")
plt.show()

In [ ]:
# Save for downstream notebooks
df_obs.to_parquet("../data/simulation_observational.parquet")
df_rct.to_parquet("../data/simulation_rct.parquet")
print("Saved simulation DataFrames.")